In [20]:
import cv2
import numpy as np
import os
import pandas as pd

input_folder = "enhanced_frames"

os.makedirs("output_combined", exist_ok=True)
os.makedirs("predicted_masks", exist_ok=True)
os.makedirs("pseudo_gt_masks", exist_ok=True)
os.makedirs("class_masks", exist_ok=True)

def compute_metrics(pred, gt):

    pred = (pred > 0).astype(np.uint8)
    gt = (gt > 0).astype(np.uint8)

    TP = np.sum((pred == 1) & (gt == 1))
    FP = np.sum((pred == 1) & (gt == 0))
    FN = np.sum((pred == 0) & (gt == 1))
    TN = np.sum((pred == 0) & (gt == 0))

    acc = (TP + TN) / (TP + TN + FP + FN + 1e-6)
    prec = TP / (TP + FP + 1e-6)
    rec = TP / (TP + FN + 1e-6)
    f1 = 2 * prec * rec / (prec + rec + 1e-6)
    iou = TP / (TP + FP + FN + 1e-6)

    return acc, prec, rec, f1, iou

def road_roi(gray):

    h, w = gray.shape

    mask = np.zeros((h, w), dtype=np.uint8)

    # road area (lower 60%)
    mask[int(h*0.35):h, :] = 255

    return mask


def detect_pothole(gray):
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(blur, 60, 150)

    kernel = np.ones((7,7), np.uint8)
    return cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, 1)

def detect_alligator(gray):
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    sobelx = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)

    mag = cv2.magnitude(sobelx, sobely)
    mag = np.uint8(np.clip(mag, 0, 255))

    _, th = cv2.threshold(mag, 50, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5,5), np.uint8)
    return cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, 1)

def detect_edge_crack(gray):
    blur = cv2.GaussianBlur(gray, (3,3), 0)
    edges = cv2.Canny(blur, 80, 200)
    return edges

def detect_crossing(gray):
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    th = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        21, 7
    )
    return th

def build_pseudo_gt(gray):

    roi = road_roi(gray)
    blur = cv2.GaussianBlur(gray, (7,7), 0)

    th = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        25, 6
    )

    th = cv2.bitwise_and(th, th, mask=roi)

    kernel = np.ones((5,5), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, 1)

    return th


results = []

files = sorted(os.listdir(input_folder))

for file in files:

    if not file.lower().endswith(('.png','.jpg','.jpeg')):
        continue

    frame = cv2.imread(os.path.join(input_folder, file))
    if frame is None:
        continue

    frame = cv2.resize(frame, (640, 480))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    name = os.path.splitext(file)[0]

    roi = road_roi(gray)

    pothole = cv2.bitwise_and(detect_pothole(gray), roi)
    alligator = cv2.bitwise_and(detect_alligator(gray), roi)
    edge = cv2.bitwise_and(detect_edge_crack(gray), roi)
    crossing = cv2.bitwise_and(detect_crossing(gray), roi)

    pred_mask = cv2.bitwise_or(pothole, alligator)
    pred_mask = cv2.bitwise_or(pred_mask, edge)
    pred_mask = cv2.bitwise_or(pred_mask, crossing)

    gt_mask = build_pseudo_gt(gray)

    cv2.imwrite(f"predicted_masks/{name}.png", pred_mask)
    cv2.imwrite(f"pseudo_gt_masks/{name}.png", gt_mask)

    cv2.imwrite(f"class_masks/{name}_pothole.png", pothole)
    cv2.imwrite(f"class_masks/{name}_alligator.png", alligator)
    cv2.imwrite(f"class_masks/{name}_edge.png", edge)
    cv2.imwrite(f"class_masks/{name}_crossing.png", crossing)

    overlay = frame.copy()
    overlay[pothole > 0] = [0, 0, 255]      # red
    overlay[alligator > 0] = [0, 255, 0]    # green
    overlay[edge > 0] = [255, 0, 0]         # blue
    overlay[crossing > 0] = [0, 255, 255]   # yellow

    combined = np.hstack((
        cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR),
        cv2.cvtColor(pred_mask, cv2.COLOR_GRAY2BGR),
        overlay
    ))

    cv2.imwrite(f"output_combined/{name}.png", combined)

    acc, prec, rec, f1, iou = compute_metrics(pred_mask, gt_mask)

    if "Pothole" in file:
        category = "Pothole"
    elif "Alligator" in file:
        category = "Alligator"
    elif "Edge" in file:
        category = "Edge Crack"
    else:
        category = "Crossing"

    results.append([category, acc, prec, rec, f1, iou])

df = pd.DataFrame(results, columns=[
    "Category", "Accuracy", "Precision", "Recall", "F1-score", "IoU"
])

table_a = df.groupby("Category").mean().reset_index()

print("\n================ TABLE A ================\n")
print(table_a)

table_a.to_csv("Table_A.csv", index=False)

print("\n Saved: Table_A.csv")


================ TABLE A ================

     Category  Accuracy  Precision    Recall  F1-score       IoU
0   Alligator  0.472051   0.177756  0.999980  0.301723  0.177755
1    Crossing  0.678153   0.259386  0.999502  0.408325  0.259346
2  Edge Crack  0.422970   0.091370  0.999920  0.166585  0.091369
3     Pothole  0.522658   0.180428  0.999910  0.301174  0.180425

 Saved: Table_A.csv
